# Scanner 3D V2

The idea is to generate a .obj file of a 3D object from the shadows of the object.

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'
from IPython.display import Image as IImage
import ipywidgets as widgets
from IPython.display import HTML, Video, display
from random import random
from panda3d.core import Triangulator
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
import plotly.graph_objects as go
import pprint
import os
import sys

root_path = '../'
sys.path.append(root_path)

from src.config import Scanner3dConfig
from src.polygon_utils import draw_multipolygon, tracesToFig
from src.mymesh import MyMesh
from src.scanner3d import load_images, processLayer, getLayers, mergetwoslices, layerstomesh

In [ ]:
config = Scanner3dConfig()
config.nb_images = 30
config.width = 250
config.height = 150
config.input_folder = "data/"
print("Config: ")
pprint.pprint(config.__dict__)
print()

### We import the shadows of the object at different angles going from 0 to PI.

In [ ]:
images = load_images(config, os.path.join(root_path, config.input_folder))

image_filenames = sorted(os.listdir(os.path.join(root_path, config.input_folder)))
html = "<div class='grid'>"
for img in image_filenames:
    html += f'<img src="{os.path.join(root_path, config.input_folder, img)}", style="width:5em" class="frame">'
html += "</div>"
HTML(html)

## Main idea: shadow intersection at each height

We will observe what happens at the layer height=60 to understand how we compute the slice of our object from the shadows at that height

In [ ]:
layer60 = processLayer(config, images, 60)

traces = []
for mpoly in layer60:
    traces += draw_multipolygon(mpoly, alpha=1/config.nb_images)
fig = tracesToFig(traces, title="Intersection of all shadows at height 60").show()

Here you can see all the shadows at different angles of the layer y=60, you can zoom on one shadow to observe its shape and you can zoom in the middle to observe the intersection of the shadows and you can see that it looks like a cross section of our object at layer y=60.

Now if we process the intersection of those shadows at each layer, we get a slice of our object at every height

In [ ]:
def get_global_bounds(multipolygons):
    """Compute the global bounding box for all multipolygons."""
    min_x = min(p.bounds[0] for mp in multipolygons for p in mp.geoms)
    min_y = min(p.bounds[1] for mp in multipolygons for p in mp.geoms)
    max_x = max(p.bounds[2] for mp in multipolygons for p in mp.geoms)
    max_y = max(p.bounds[3] for mp in multipolygons for p in mp.geoms)
    return min_x, min_y, max_x, max_y

In [ ]:
multipolygons = getLayers(config, images)
global_min_x, global_min_y, global_max_x, global_max_y = get_global_bounds(multipolygons)

frames = [
    go.Frame(
        data=draw_multipolygon(multipolygons[frame], alpha=1),
        name=str(frame)
    )
    for frame in range(len(multipolygons))
]
max_traces = max(len(f.data) for f in frames)
normalized = []
for f in frames:
    data = list(f.data)
    while len(data) < max_traces:
        data.append(go.Scatter(x=[], y=[], visible=False, showlegend=False))
    normalized.append(go.Frame(data=data, name=f.name))
frames = normalized

fig = go.Figure(
    data=frames[60].data,
    layout=go.Layout(
        xaxis=dict(
            range=[-100, 100],
            scaleanchor="y"
        ),
        yaxis=dict(
            range=[-100, 100]
        ),
        xaxis_title="x",
        yaxis_title="y",
        sliders=[{
            "steps": [
                {
                    "method": "animate",
                    "args": [[str(i)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}],
                    "label": str(i)
                }
                for i in range(len(frames))
            ],
            "transition": {"duration": 0},
            "x": 0, "y": 0,
            "currentvalue": {"prefix": "Height: "}
        }]
    ),
    frames=frames
)

fig.show()

## Construct the faces for the .obj file

Now let's make a mesh out of those slices, we make a MyMesh class to easily add faces and plot it.

### Vertical faces (correspond to one slice)
If one slice is polygon (a, b, c) we construct the vertical faces by simply extruding the polygon.

In [ ]:
testmesh = MyMesh()
pta = (0, 0)
a1 = testmesh.addVertex(pta)
ptb = (1, 0)
b1 = testmesh.addVertex(ptb)
ptc = (0, 1)
c1 = testmesh.addVertex(ptc)
print(f"Points a1: {pta}, b1: {ptb}, c1: {ptc}")

a2 = testmesh.addVertex(pta)
b2 = testmesh.addVertex(ptb)
c2 = testmesh.addVertex(ptc)
print(f"Points a2: {pta}, b2: {ptb}, c2: {ptc}")

print(f"Set height of a1, b1, c1 at 0")
testmesh.setpointsheight([a1, b1, c1], 0)
print(f"Set height of a2, b2, c2 at 1")
testmesh.setpointsheight([a2, b2, c2], 1)

testmesh.addFaceToFace([(a1, pta), (b1, ptb), (c1, ptc)], [(a2, pta), (b2, ptb), (c2, ptc)], Polygon([(0, 0), (1, 0), (0, 1)]))
print(f"Mesh faces: {testmesh.faces}")
testmesh.plot()

### Horizontal faces (between two slices)

If one slice has polygon A and the next slice has polygon B, we take A\B's faces up and B\A's faces down.

Here is one clear example.

In [ ]:
meshexemple = MyMesh()

poly1 = MultiPolygon([Polygon([(0, 0), (2, 0), (2, 2), (0, 2)]), Polygon([(4,0), (5, 0), (5, 2), (4, 2)])])
poly2 = MultiPolygon([Polygon([(1, 1.5), (5, 0), (4, 0.5), (4, 1), (3, 1.5), (4, 2)]), Polygon([(0.5, 0.5), (0.7, 0.5), (0.7, 0.7)])])

traces1 = draw_multipolygon(poly1, color='green', alpha=0.1)
traces2 = draw_multipolygon(poly2, color='blue', alpha=0.1)

pts, traces3et4 = mergetwoslices(meshexemple, poly1, poly2, 60.5, visualize=True)
fig = tracesToFig(traces1+traces2+traces3et4).show()

Real example with the layers 60 and 61.

In [ ]:
layer = 60
poly1 = multipolygons[layer]
poly2 = multipolygons[layer+1]

traces = []
traces += draw_multipolygon(poly1, color='green', alpha=0.1)
traces += draw_multipolygon(poly2, color='blue', alpha=0.1)
pts, tracetoAdd = mergetwoslices(meshexemple, poly1, poly2, 60.5, visualize=True)
traces += tracetoAdd
tracesToFig(traces,title="Horizontal faces between layers 60 and 61")

## Result

In [ ]:
mesh = layerstomesh(config, multipolygons)
mesh.plot()